## Лабораторная работа № 1
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1. Подготовка датасета Delivery ETA

Прочитаем датасет из HDFS, проверим структуру и типы признаков, выполним преобразования и сохраним результат в Apache Iceberg.

Объект наблюдения — потенциальный заказ в сервисе онлайн-доставки продуктов. Целевая переменная `delivery_eta_minutes` — время прибытия курьера в минутах. Описание признаков — в файле `description_data_set.md`. Исходный файл `delivery_eta.parquet` занимает около 10,3 ГиБ.

Ноутбук запускается на сервере с Spark 3.5 / Scala 2.12 и YARN. До первого запуска файл нужно загрузить из локальной папки удалённого сервера в HDFS. Выполните в **терминале сервера**, находясь в папке с ноутбуками:

```bash
# Проверяем HDFS и доступное место.
hdfs dfs -ls /
hdfs dfs -df -h

# Загружаем файл из локальной папки удалённого сервера в HDFS.
hdfs dfs -mkdir -p /user/$USER/datasets
hdfs dfs -copyFromLocal delivery_eta.parquet /user/$USER/datasets/

# Проверяем результат загрузки.
hdfs dfs -ls -h /user/$USER/datasets/delivery_eta.parquet
```

Эта загрузка выполняется один раз. Если файл уже находится в HDFS по указанному адресу, повторять команды не нужно. Вторая часть читает таблицу Iceberg, сохранённую этим ноутбуком.


Подключим необходимые библиотеки. Для преобразований используем PySpark DataFrame API.


In [1]:
import os  # Имя пользователя из окружения операционной системы.
from pyspark import SparkConf  # Настройки ресурсов Spark.
from pyspark.sql import SparkSession, DataFrame  # Сессия и распределённая таблица.
from pyspark.sql import functions as F  # Функции обработки столбцов.


Укажем адрес предварительно загруженного файла в HDFS и имена базы и таблицы. Имя базы должно совпадать с указанным во второй части. При повторной записи содержимое таблицы заменяется.


In [2]:
# Имя текущего пользователя сервера для путей HDFS.
user_name = os.getenv("USER")

# Адрес датасета, заранее загруженного в HDFS.
path = f"hdfs:///user/{user_name}/datasets/delivery_eta.parquet"

# Имена базы и таблицы для подготовленных данных.
database_name = "delivery_eta_database"
table_name = "sobd_lab1_delivery_eta"
target_col = "delivery_eta_minutes"  # Время прибытия курьера в минутах.
timestamp_col = "timestamp"  # Момент формирования заказа.
full_table_name = f"spark_catalog.{database_name}.{table_name}"


Настроим Spark для выполнения на YARN: два исполнителя по 8 ядер и 12 ГБ памяти. Подключим Iceberg и зададим пользовательское хранилище таблиц в HDFS.


In [3]:
# Объявляем функцию без аргументов. -> SparkConf — подсказка о типе возвращаемого результата.
def create_spark_configuration() -> SparkConf:
    """Настройки Spark 3.5 и Iceberg для учебного кластера."""
    # Создаём пустой набор настроек и ниже заполняем его.
    conf = SparkConf()
    # Имя приложения, по которому его можно найти в Spark UI и YARN.
    conf.setAppName("lab 1 Delivery ETA")
    # YARN распределяет ресурсы Hadoop-кластера; вычисления будут выполняться на кластере.
    conf.setMaster("yarn")
    # Режим client: драйвер управляет работой из процесса ноутбука, исполнители выполняют задачи на кластере.
    conf.set("spark.submit.deployMode", "client")
    # Каждому исполнителю запрашиваем 12 ГБ оперативной памяти.
    conf.set("spark.executor.memory", "12g")
    # Каждому исполнителю выделяем 8 процессорных ядер.
    conf.set("spark.executor.cores", "8")
    # Запрашиваем 2 исполнителя: вместе это 16 ядер; фактическое выделение зависит от кластера.
    conf.set("spark.executor.instances", "2")
    # Запрашиваем 4 ГБ для драйвера. В client-режиме лимит памяти JVM задают до её запуска.
    conf.set("spark.driver.memory", "4g") # память драйвера: 4 ГБ; он принимает результаты collect() и toPandas(), поэтому не стоит просить у него слишком много данных
    conf.set("spark.driver.cores", "2")
    # Подключаем Iceberg 1.6.0 для Spark 3.5 и Scala 2.12: библиотека нужна для чтения и записи таблиц.
    conf.set("spark.jars.packages",
             "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    # Включаем расширения Spark SQL для работы с Iceberg.
    conf.set("spark.sql.extensions",
             "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Назначаем каталогу spark_catalog реализацию Iceberg, которая знает формат этих таблиц.
    conf.set("spark.sql.catalog.spark_catalog",
             "org.apache.iceberg.spark.SparkCatalog")
    # Метаданные таблиц будут храниться в файловой системе, без отдельного сервера каталога.
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    # Папка в HDFS для файлов и метаданных наших таблиц; имя пользователя подставится автоматически.
    conf.set("spark.sql.catalog.spark_catalog.warehouse",
             f"hdfs:///user/{user_name}/warehouse")
    # Указываем класс, через который Iceberg будет читать и записывать файлы Hadoop/HDFS.
    conf.set("spark.sql.catalog.spark_catalog.io-impl",
             "org.apache.iceberg.hadoop.HadoopFileIO")
    # Возвращаем готовые настройки туда, откуда была вызвана функция.
    return conf

# Вызываем функцию: теперь переменная conf хранит все перечисленные настройки.
conf = create_spark_configuration()


Создадим сессию Spark. Если в ядре уже работает сессия с другими настройками, сначала остановите её или перезапустите ядро.


In [4]:
# Передаём настройки в конструктор сессии; создаём сессию или получаем уже существующую.
# Существующая сессия может сохранить прежние настройки: для смены конфигурации лучше перезапустить ядро.
spark = SparkSession.builder.config(conf=conf).getOrCreate()
# Последняя строка ячейки выводит сведения о сессии: версию Spark, имя приложения и ссылку на Spark UI.
spark


:: loading settings :: url = jar:file:/opt/spark-3.5.2-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/user0/.ivy2/cache
The jars for the packages stored in: /home/user0/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6af11029-01e5-4b0d-82b4-f66945c42254;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 in central
:: resolution report :: resolve 693ms :: artifacts dl 20ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spa

### Чтение данных из HDFS

Прочитаем предварительно загруженный файл в датафрейм Spark. Parquet хранит названия и типы столбцов, поэтому отдельно задавать схему не требуется.

Число строк и диапазон целевой переменной вычислим ниже на загруженных данных. Короткий файл `delivery_eta.csv` — только образец строк, расчёты выполняем по полному Parquet.


In [5]:
# Читаем Parquet из HDFS в распределённую таблицу Spark.
df = spark.read.parquet(path)
df.show()


26/10/28 19:23:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+-----------+-----------+-----------+-----------+-----------+------------+-----------+------------+-----------+------------+-------------+------------+------------+------------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+------------+-------------+------------+-----------+-----------+-----------+------------+------------+-----------+-----------+-------------+-------------+-----------+-------------+-------------+-----------+-----------+------------+-----------+------------+-----------+-------------+-----------+-----------+------------+------------+------------+-----------+------------+-----------+-----------+-----------+------------+-----------+-------------+------------+-----------+-----------+------------+------------+-----------+------------+-------------+-----------+-----------+------------+-----------+-----------+------------+-------------+------------+----------+------------+-------------+-----------+------------+-----------+----

### Содержание датасета и назначение столбцов

Каждая строка описывает потенциальный заказ в сервисе онлайн-доставки продуктов. Задача — прогнозировать, через сколько минут прибудет курьер. Признаки обезличены: они агрегируют доступность курьеров, гео- и навигационные оценки, историю предыдущих доставок и характеристики времени суток.

| Название столбца | Расшифровка |
|---|---|
| `num_0` … `num_220` | 221 обезличенный стандартизованный числовой признак. Значения обычно около нуля, во многих столбцах есть пропуски |
| `bin_0` | Обезличенный бинарный признак со значениями 0 и 1. Точная семантика не раскрыта |
| `cat_0` | Первый обезличенный категориальный признак. Значения — числовые коды категорий |
| `cat_1` | Второй обезличенный категориальный признак. Значения — числовые коды категорий |
| `cat_2` | Третий обезличенный категориальный признак. Значения — числовые коды категорий |
| `timestamp` | Дата и время формирования заказа. Используется для анализа по времени и хронологического разбиения данных |
| `delivery_eta_minutes` | Время прибытия курьера в минутах. Целевая переменная для прогнозирования |

Всего в исходном файле **227 столбцов**: 225 входных признаков, временная метка и целевая переменная.

Числовые коды `cat_*` обозначают категории, поэтому не трактуем их как количественные величины. Среднее арифметическое по коду зоны или типа заказа предметного смысла не имеет.


In [6]:
# Создаём пустые списки названий признаков.
num_cols = []
bin_cols = []
cat_cols = []

# По очереди смотрим название каждого столбца.
for column in df.columns:
    if column.startswith("num_"):
        num_cols.append(column)
    elif column.startswith("bin_"):
        bin_cols.append(column)
    elif column.startswith("cat_"):
        cat_cols.append(column)

print(f"Всего столбцов: {len(df.columns)}")
print(f"Числовых признаков: {len(num_cols)}")
print(f"Бинарных признаков: {len(bin_cols)}")
print(f"Категориальных признаков: {len(cat_cols)}")

# Для просмотра берём первые пять числовых признаков и остальные группы.
preview_cols = num_cols[:5] + bin_cols + cat_cols + [timestamp_col, target_col]
df.select(preview_cols).show(5, truncate=30)


Всего столбцов: 227
Числовых признаков: 221
Бинарных признаков: 1
Категориальных признаков: 3


+-----------+----------+-----------+-----------+----------+-----+-----+-----+-----+-------------------+--------------------+
|      num_0|     num_1|      num_2|      num_3|     num_4|bin_0|cat_0|cat_1|cat_2|          timestamp|delivery_eta_minutes|
+-----------+----------+-----------+-----------+----------+-----+-----+-----+-----+-------------------+--------------------+
| -1.3938484|-0.6241448|-0.24972141| 0.28091082|-1.3943632|  0.0|    1|    1|    1|2023-11-25 19:01:37|           1.8833333|
|  0.3448193| 1.1140325|  1.1210603|0.098013006|0.35988075|  1.0|    2|    2|    2|2023-12-12 12:12:12|           6.4333334|
| -1.6411444|-1.6128669|-0.77483284|  1.2339616|-1.6322846|  1.0|    3|    3|    3|2023-12-14 11:07:59|           5.0666666|
|  -0.637411|0.48177853| 0.71846205|-0.26528385|-0.5761903|  1.0|    4|    4|    4|2023-10-30 09:25:34|           7.7166667|
|0.053135037|0.39452538| 0.26788375| -1.1234151|0.08424524|  0.0|    5|    5|    5|2023-12-17 16:22:38|           17.316668|


Выведем схему исходного датафрейма. Полный набор из 227 столбцов сохраняется, хотя для просмотра выше выбрана только часть.


In [7]:
# Печатаем схему: имена и типы столбцов. nullable = true означает, что в столбце допускаются пропуски.
df.printSchema()


root
 |-- num_0: float (nullable = true)
 |-- num_1: float (nullable = true)
 |-- num_2: float (nullable = true)
 |-- num_3: float (nullable = true)
 |-- num_4: float (nullable = true)
 |-- num_5: float (nullable = true)
 |-- num_6: float (nullable = true)
 |-- num_7: float (nullable = true)
 |-- num_8: float (nullable = true)
 |-- num_9: float (nullable = true)
 |-- num_10: float (nullable = true)
 |-- num_11: float (nullable = true)
 |-- num_12: float (nullable = true)
 |-- num_13: float (nullable = true)
 |-- num_14: float (nullable = true)
 |-- num_15: float (nullable = true)
 |-- num_16: float (nullable = true)
 |-- num_17: float (nullable = true)
 |-- num_18: float (nullable = true)
 |-- num_19: float (nullable = true)
 |-- num_20: float (nullable = true)
 |-- num_21: float (nullable = true)
 |-- num_22: float (nullable = true)
 |-- num_23: float (nullable = true)
 |-- num_24: float (nullable = true)
 |-- num_25: float (nullable = true)
 |-- num_26: float (nullable = true)
 |-- n

Если `timestamp` сохранён строкой в формате ISO, приведём его к типу timestamp. Числовые, бинарные и категориальные столбцы на этом шаге не меняем.


In [8]:
# Смотрим фактический тип временной метки в прочитанном Parquet.
timestamp_type = dict(df.dtypes)[timestamp_col]
print(f"Тип {timestamp_col}: {timestamp_type}")

if timestamp_type == "string":
    df = df.withColumn(timestamp_col, F.to_timestamp(F.col(timestamp_col)))
    print("Строковая метка преобразована в timestamp.")
else:
    print("Преобразование метки не требуется.")


Тип timestamp: timestamp_ntz
Преобразование метки не требуется.


### Проверка значений до обработки

Схема показывает типы, но не доказывает, что значения корректны. До изменений проверим:

* какие значения встречаются в `bin_*` и есть ли среди них что-либо кроме 0, 1 и пропусков;
* есть ли `NaN` и бесконечности в `num_*` и целевом столбце.

Обычные `NULL` считаем отдельно ниже: отсутствие значения не является причиной менять тип столбца.


In [9]:
# Схему типов уже показали выше. Теперь смотрим сами бинарные значения.
binary_counts = {}
binary_problem_cols = []

for column in bin_cols:
    # Сначала группируем одинаковые значения, затем считаем их количество.
    value_counts = df.groupBy(column).count()
    value_counts = value_counts.orderBy(column)
    value_counts.show(10, truncate=False)

    # Исключаем пропуски: пропуск не является недопустимым бинарным значением.
    non_empty_values = df.filter(F.col(column).isNotNull())

    # Оставляем строки, где значение не равно 0.
    not_zero_values = non_empty_values.filter(F.col(column) != 0)

    # Из оставшихся строк исключаем также значение 1.
    # Теперь здесь только значения, которые не равны ни 0, ни 1.
    invalid_values = not_zero_values.filter(F.col(column) != 1)
    invalid_count = invalid_values.count()

    binary_counts[column] = invalid_count
    print(f"{column}: значений вне 0 и 1 — {invalid_count}")

    if invalid_count > 0:
        binary_problem_cols.append(column)


+-----+--------+
|bin_0|count   |
+-----+--------+
|0.0  |5411416 |
|1.0  |11632627|
+-----+--------+



bin_0: значений вне 0 и 1 — 0


In [10]:
# Числовые признаки и целевой столбец. Категории и бинарный признак сюда не входят.
numeric_columns = num_cols + [target_col]

# Список выражений для одного общего подсчёта.
aggregates = []
for column in numeric_columns:
    value = F.col(column)
    # count считает только непустые результаты when, поэтому NULL в сумму не попадают.
    aggregates.append(F.count(F.when(F.isnan(value), 1)).alias(column + "_nan"))
    aggregates.append(F.count(F.when(value == float("inf"), 1)).alias(column + "_pos_inf"))
    aggregates.append(F.count(F.when(value == float("-inf"), 1)).alias(column + "_neg_inf"))

# Один agg читает таблицу один раз и сразу считает все три показателя по каждому столбцу.
counts = df.agg(*aggregates).first().asDict()

numeric_problem_cols = []
for column in numeric_columns:
    nan_count = counts[column + "_nan"]
    positive_inf_count = counts[column + "_pos_inf"]
    negative_inf_count = counts[column + "_neg_inf"]
    invalid_count = nan_count + positive_inf_count + negative_inf_count

    # В список попадают только столбцы, где проблема действительно есть.
    if invalid_count > 0:
        numeric_problem_cols.append(column)
        print(f"{column}: NaN — {nan_count}")
        print(f"{column}: +inf — {positive_inf_count}, -inf — {negative_inf_count}")

print(f"Числовых столбцов с NaN или бесконечностями: {len(numeric_problem_cols)}")

num_10: NaN — 11238
num_10: +inf — 0, -inf — 0
num_11: NaN — 11238
num_11: +inf — 0, -inf — 0
num_76: NaN — 9517563
num_76: +inf — 0, -inf — 0
num_77: NaN — 9517563
num_77: +inf — 0, -inf — 0
num_78: NaN — 9517563
num_78: +inf — 0, -inf — 0
num_79: NaN — 9517563
num_79: +inf — 0, -inf — 0
num_80: NaN — 9517563
num_80: +inf — 0, -inf — 0
num_81: NaN — 10129523
num_81: +inf — 0, -inf — 0
num_82: NaN — 10129523
num_82: +inf — 0, -inf — 0
num_83: NaN — 10129523
num_83: +inf — 0, -inf — 0
num_84: NaN — 10129523
num_84: +inf — 0, -inf — 0
num_85: NaN — 10129523
num_85: +inf — 0, -inf — 0
num_86: NaN — 9580071
num_86: +inf — 0, -inf — 0
num_87: NaN — 9580071
num_87: +inf — 0, -inf — 0
num_88: NaN — 9580071
num_88: +inf — 0, -inf — 0
num_89: NaN — 9580071
num_89: +inf — 0, -inf — 0
num_90: NaN — 9580071
num_90: +inf — 0, -inf — 0
num_91: NaN — 9517563
num_91: +inf — 0, -inf — 0
num_92: NaN — 9517563
num_92: +inf — 0, -inf — 0
num_93: NaN — 9517563
num_93: +inf — 0, -inf — 0
num_94: NaN — 95175

### Решение по результатам проверки

**Тип и качество значения — разные вещи.** Запись `1.0` в бинарном признаке не является ошибкой: она корректно обозначает одно из двух состояний.

| Что наблюдаем | Решение |
|---|---|
| `num_*` и `delivery_eta_minutes` уже имеют тип float | Сохраняем тип |
| `bin_*` содержат только 0, 1 и NULL | Сохраняем тип |
| В `bin_*` обнаружены другие значения | Заменяем только недопустимые значения на NULL, сохраняя тип столбца |
| `cat_*` уже имеют целочисленный тип | Оставляем |
| `timestamp` имеет тип timestamp или приведён к нему | Оставляем |
| В числовом столбце обнаружены NaN или бесконечности | Заменяем их на NULL только в тех столбцах, где проверка действительно выявила проблему |

`NaN` означает «не число». Их замена на NULL позволяет учитывать такие значения как пропуски при дальнейшей очистке.


In [11]:
# Функция получает таблицу и возвращает таблицу после необходимых изменений.
def transform_dataframe(data: DataFrame) -> DataFrame:
    # Множества нужны, чтобы быстро проверять, надо ли менять столбец.
    problem_numeric = set(numeric_problem_cols)
    problem_binary = set(binary_problem_cols)
    columns = []

    for column in data.columns:
        value = F.col(column)

        if column in problem_numeric:
            # Оставляем обычные числа. NaN и бесконечности становятся NULL.
            # when без otherwise возвращает пропуск и сохраняет тип столбца.
            cleaned = F.when(
                ~F.isnan(value) & (value != float("inf")) & (value != float("-inf")),
                value,
            )
            columns.append(cleaned.alias(column))

        elif column in problem_binary:
            # 0 и 1 оставляем, любое другое значение заменяем пропуском.
            cleaned = F.when(value.isin(0, 1), value)
            columns.append(cleaned.alias(column))

        else:
            # Столбцы без проблем не трогаем.
            columns.append(value)

    # Один select вместо сотен withColumn: план строится один раз.
    return data.select(*columns)

Применим только изменения, обоснованные диагностикой выше. Функция сохраняет исходные типы всех столбцов.


In [12]:
# Выполняем подготовку после изучения диагностических таблиц.
df = transform_dataframe(df)


In [13]:
# Сравниваем фрагмент данных и схему с исходным выводом.
df.select(preview_cols).show(5, truncate=30)
df.printSchema()


+-----------+----------+-----------+-----------+----------+-----+-----+-----+-----+-------------------+--------------------+
|      num_0|     num_1|      num_2|      num_3|     num_4|bin_0|cat_0|cat_1|cat_2|          timestamp|delivery_eta_minutes|
+-----------+----------+-----------+-----------+----------+-----+-----+-----+-----+-------------------+--------------------+
| -1.3938484|-0.6241448|-0.24972141| 0.28091082|-1.3943632|  0.0|    1|    1|    1|2023-11-25 19:01:37|           1.8833333|
|  0.3448193| 1.1140325|  1.1210603|0.098013006|0.35988075|  1.0|    2|    2|    2|2023-12-12 12:12:12|           6.4333334|
| -1.6411444|-1.6128669|-0.77483284|  1.2339616|-1.6322846|  1.0|    3|    3|    3|2023-12-14 11:07:59|           5.0666666|
|  -0.637411|0.48177853| 0.71846205|-0.26528385|-0.5761903|  1.0|    4|    4|    4|2023-10-30 09:25:34|           7.7166667|
|0.053135037|0.39452538| 0.26788375| -1.1234151|0.08424524|  0.0|    5|    5|    5|2023-12-17 16:22:38|           17.316668|


Проверим количество строк, диапазон времени заказа и целевой переменной.


In [14]:
# Считаем строки в таблице.
row_count = df.count()
print(f"Количество строк: {row_count}")

# select() выбирает результат расчёта.
# first() получает единственную строку результата, [0] — значение в ней.
first_time = df.select(F.min(timestamp_col)).first()[0]
last_time = df.select(F.max(timestamp_col)).first()[0]
print(f"Период наблюдений: {first_time} — {last_time}")

min_eta = df.select(F.min(target_col)).first()[0]
max_eta = df.select(F.max(target_col)).first()[0]
print(f"Минимальное время прибытия: {min_eta}")
print(f"Максимальное время прибытия: {max_eta}")

# Сначала отбираем нужные строки, затем считаем их.
negative_time_rows = df.filter(F.col(target_col) < 0)
negative_time_count = negative_time_rows.count()
print(f"Строк с отрицательным временем прибытия: {negative_time_count}")

missing_target_rows = df.filter(F.col(target_col).isNull())
missing_target_count = missing_target_rows.count()
print(f"Пропусков времени прибытия: {missing_target_count}")

missing_timestamp_rows = df.filter(F.col(timestamp_col).isNull())
missing_timestamp_count = missing_timestamp_rows.count()
print(f"Пропусков временной метки: {missing_timestamp_count}")


Количество строк: 17044043


Период наблюдений: 2023-10-20 00:10:02 — 2024-01-07 22:53:09


Минимальное время прибытия: 0.0
Максимальное время прибытия: 170.51666259765625


Строк с отрицательным временем прибытия: 0


Пропусков времени прибытия: 0


Пропусков временной метки: 0


Число строк, период и диапазон цели уже посчитаны выше. Отдельный подсчёт пропусков по всем 227 столбцам здесь не нужен: на запись таблицы он не влияет. Пропуски, их удаление и заполнение делает вторая часть — это пункт задания про разведочный анализ.

Сохраним подготовленный датафрейм в Apache Iceberg. Создадим базу данных в каталоге Spark.


In [15]:
# Передаём SQL-команду Spark: создаём базу, если её ещё нет; существующая база сохраняется.
spark.sql(f"CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}")
# Выбираем базу текущей: короткие имена таблиц будут искаться в ней.
spark.catalog.setCurrentDatabase(database_name)


Запишем таблицу.


In [16]:
# writeTo() указывает, в какую таблицу записать df; using() задаёт формат Iceberg.
# createOrReplace() создаёт таблицу или заменяет существующую. Здесь реально запускаются вычисления и запись.
df.writeTo(full_table_name).using("iceberg").createOrReplace()


26/10/28 19:37:11 WARN HadoopTableOperations: Error reading version hint file hdfs:/user/user0/warehouse/delivery_eta_database/sobd_lab1_delivery_eta/metadata/version-hint.text
java.io.FileNotFoundException: File does not exist: /user/user0/warehouse/delivery_eta_database/sobd_lab1_delivery_eta/metadata/version-hint.text
	at org.apache.hadoop.hdfs.server.namenode.INodeFile.valueOf(INodeFile.java:87)
	at org.apache.hadoop.hdfs.server.namenode.INodeFile.valueOf(INodeFile.java:77)
	at org.apache.hadoop.hdfs.server.namenode.FSDirStatAndListingOp.getBlockLocations(FSDirStatAndListingOp.java:159)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.getBlockLocations(FSNamesystem.java:2198)
	at org.apache.hadoop.hdfs.server.namenode.NameNodeRpcServer.getBlockLocations(NameNodeRpcServer.java:795)
	at org.apache.hadoop.hdfs.protocolPB.ClientNamenodeProtocolServerSideTranslatorPB.getBlockLocations(ClientNamenodeProtocolServerSideTranslatorPB.java:468)
	at org.apache.hadoop.hdfs.protocol.proto

Проверим наличие таблицы и прочитаем небольшой фрагмент уже из Iceberg.


In [17]:
# Показываем список таблиц базы и проверяем, что наша таблица появилась.
spark.sql(f"SHOW TABLES IN spark_catalog.{database_name}").show(truncate=False)
# Читаем сохранённую таблицу по её полному имени в новый DataFrame.
saved_df = spark.table(full_table_name)
# Выводим 5 строк из сохранённой таблицы, чтобы проверить результат записи.
saved_df.select(preview_cols).show(5, truncate=30)


+---------------------+----------------------+-----------+
|namespace            |tableName             |isTemporary|
+---------------------+----------------------+-----------+
|delivery_eta_database|sobd_lab1_delivery_eta|false      |
+---------------------+----------------------+-----------+



+------------+----------+-----------+----------+------------+-----+-----+-----+-----+-------------------+--------------------+
|       num_0|     num_1|      num_2|     num_3|       num_4|bin_0|cat_0|cat_1|cat_2|          timestamp|delivery_eta_minutes|
+------------+----------+-----------+----------+------------+-----+-----+-----+-----+-------------------+--------------------+
|-0.005018293|0.34681705| 0.22005822|-1.1971933|-0.062015414|  0.0|  240|  249|  249|2023-10-22 05:33:58|                2.75|
|  -1.4215561|-0.6611619|-0.32491526| 0.3255765|  -1.4291298|  1.0|  103|  131|  131|2023-12-25 09:05:20|           0.8666667|
|  0.13148026|0.66741896|  0.6356186|-1.2612822|  0.15100867|  1.0|  303|  316|  316|2023-11-13 14:02:21|                 9.8|
|  0.73812836|-0.8808932| -0.9830647|0.47100437|   0.7496954|  1.0|  476|  508|  508|2023-12-21 18:08:18|                12.9|
| -0.47802472|0.58399165|  0.7529975|-0.4696031| -0.50353813|  0.0|  191|  197|  197|2023-11-18 10:49:51|      

После завершения работы остановим сессию Spark.


In [18]:
# Завершаем сессию Spark и освобождаем ресурсы приложения на кластере.
# Для дальнейших вычислений после этой команды нужно снова создать сессию и прочитать данные.
spark.stop()
